In [35]:
import pandas as pd
import geopandas as gpd
import numpy as np

from tqdm import tqdm
from shapely import Polygon, make_valid, geometry

In [36]:
ressource_survey_path = "../../../resources/surveys/edgt_lyon"
cleaned_survey_path = "../../../results/surveys/edgt_lyon"
locations_path = "../../../resources/locations"

output_path = "../../../results/surveys/edgt_lyon/trips.geoparquet"

In [37]:
if "papermill" in locals():
    survey_path = papermill.input["survey"]
    spatial_path = papermill.input["spatial"]

    output_path = papermill.output[0]

In [38]:
# Load spatial data

gdf_housing: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_housing.gpkg" % locations_path, layer="housing")
gdf_work: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_work.gpkg" % locations_path, layer="work")
gdf_education: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_education.gpkg" % locations_path, layer="education")
gdf_secondary: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_secondary.gpkg" % locations_path, layer="secondary")

gdf_zones: gpd.GeoDataFrame = gpd.read_file("%s/EDGT_AML2015_ZF_GT.TAB" % ressource_survey_path).to_crs("EPSG:2154")

In [39]:
gdf_housing


,location_id,weight,geometry
0,home_0,1.0,POINT (848193.38 6563109.52)
1,home_1,1.0,POINT (848203.98 6563089.67)
2,home_2,1.0,POINT (848214.41 6563076.71)
3,home_3,1.0,POINT (848221.98 6562965.48)
4,home_4,0.5,POINT (848245.17 6562942.05)
...,...,...,...
1202292,home_1202292,1.0,POINT (914706.049 6455313.756)
1202293,home_1202293,1.0,POINT (844485.142 6521425.826)
1202294,home_1202294,1.0,POINT (842684.713 6518921.219)
1202295,home_1202295,1.0,POINT (843266.058 6519578.269)


In [40]:
df_households = pd.read_parquet("%s/households.parquet" % cleaned_survey_path)
df_households

,edgt_household_id,zone_id,household_id,number_of_cars,number_of_motorbikes,number_of_bicycles
0,10100184,101001,0,0,0,2
1,101001115,101001,1,1,0,1
2,1010022,101002,2,1,0,2
3,1010024,101002,3,0,0,0
4,1010025,101002,4,0,0,0
...,...,...,...,...,...,...
6610,712451647,712451,16356,1,0,0
6611,712451653,712451,16357,1,0,1
6612,712451686,712451,16358,1,0,3
6613,712451742,712451,16359,1,2,0


In [41]:
# Merge df_households with gdf_zones to get the geometries
gdf_zones["zone_id"] = gdf_zones["ZF2015_Nouveau_codage"].astype(int)


In [42]:
df_persons: pd.DataFrame = pd.read_parquet("%s/persons.parquet" % cleaned_survey_path)
df_trips: pd.DataFrame = pd.read_parquet("%s/trips.parquet" % cleaned_survey_path)

In [43]:
df_trips


,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_cell,destination_cell,origin_activity_type,destination_activity_type,is_valid
0,0,0,0,pt,1110,900.0,38700.0,101001,102001,home,other,True
1,0,0,1,pt,1110,900.0,42300.0,102001,101001,other,home,True
2,0,1,2,pt,2590,1200.0,30000.0,101001,104001,home,work,True
3,0,1,3,pt,2750,1500.0,63900.0,104001,101001,work,home,True
4,0,2,4,pt,2390,600.0,26880.0,101001,212003,home,education,True
...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,770,480.0,27420.0,712001,712451,other,work,True


In [44]:
df_trips.rename({
    "origin_cell": "origin_zone_id",
    "destination_cell": "destination_zone_id",
    "origin_activity": "origin_activity_id",
}, axis=1, inplace=True)

df_trips

,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid
0,0,0,0,pt,1110,900.0,38700.0,101001,102001,home,other,True
1,0,0,1,pt,1110,900.0,42300.0,102001,101001,other,home,True
2,0,1,2,pt,2590,1200.0,30000.0,101001,104001,home,work,True
3,0,1,3,pt,2750,1500.0,63900.0,104001,101001,work,home,True
4,0,2,4,pt,2390,600.0,26880.0,101001,212003,home,education,True
...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,770,480.0,27420.0,712001,712451,other,work,True


In [45]:

if False:
    df_trips = df_trips.loc[:100].copy()

# Create a cache to store geometry for each person_id, zone_id, and activity
geometry_cache = {}

def get_cached_geometry(person_id, zone_id, activity):
    return geometry_cache.get((person_id, zone_id, activity))

def set_cached_geometry(person_id, zone_id, activity, geometry):
    geometry_cache[(person_id, zone_id, activity)] = geometry

# Function to select a random point based on weight
def process_trip_group(trip_group: pd.DataFrame) -> pd.Series:
    
    zone_id = trip_group.name[0]
    activity = trip_group.name[1]

    result = pd.Series([None] * len(trip_group), name="geometry")

    group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type != 'Point')]
    if group_zone.empty:
        group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type == 'Point')]
        if group_zone.empty:
            return result
        else:
            return  pd.Series([group_zone["geometry"].iloc[0]] * len(trip_group), name="geometry")

    group_zone_geometry = group_zone["geometry"].iloc[0]

    if not group_zone_geometry.is_valid:
        group_zone_geometry = make_valid(group_zone_geometry)

    weights = None

    if activity == "home":
        gdf_filtered: gpd.GeoDataFrame = gdf_housing[gdf_housing.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_housing.iloc[[gdf_housing.distance(group_zone_geometry).idxmin()]]
        weights = "weight"
    if activity == "work":
        gdf_filtered: gpd.GeoDataFrame = gdf_work[gdf_work.within(group_zone_geometry)]
        weights = "employees"
        if gdf_filtered.empty:
            gdf_filtered = gdf_work.iloc[[gdf_work.distance(group_zone_geometry).idxmin()]]
    elif activity == "education":
        gdf_filtered: gpd.GeoDataFrame = gdf_education[gdf_education.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_education.iloc[[gdf_education.distance(group_zone_geometry).idxmin()]]
        weights = "weight"
    elif activity in ["leisure", "shop", "other"]:
        mask = gdf_secondary.within(group_zone_geometry)
        mask &= gdf_secondary["activity_type"] == activity
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[mask]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]

    if gdf_filtered.empty:
        weights=None
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[gdf_secondary.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]
    
    result = gdf_filtered.sample(n=len(trip_group), weights=weights, replace=True)["geometry"].reset_index(drop=True)
    result = result.rename("geometry")

    # Use and/or update cache
    for idx in range(len(trip_group)):
        trip = trip_group.iloc[idx]
        person_id = trip["person_id"]
        cached_geometry = get_cached_geometry(person_id, zone_id, activity)
        if cached_geometry is None:
            set_cached_geometry(person_id, zone_id, activity, result.iloc[idx])
        else:
            result.iloc[idx] = cached_geometry
            pass

    return result

# NEEEED to sort first !!!
tqdm.pandas(desc="Calculating origin_geometry")
df_trips.sort_values(["origin_zone_id", "origin_activity_type"], inplace=True)
origin_geometry = df_trips.groupby(["origin_zone_id", "origin_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["origin_geometry"] = origin_geometry.values

tqdm.pandas(desc="Calculating destination_geometry")
df_trips.sort_values(["destination_zone_id", "destination_activity_type"], inplace=True)
destination_geometry = df_trips.groupby(["destination_zone_id", "destination_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["destination_geometry"] = destination_geometry.values

df_trips

Calculating destination_geometry: 100%|██████████| 7277/7277 [11:42<00:00, 10.36it/s] 


,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,origin_geometry,destination_geometry
80797,12383,27096,80797,walk,290,300.0,45000.0,101001,101001,leisure,education,True,POINT (841519.35 6517440.14),POINT (841505.6435842451 6517298.579645115)
91,19,33,91,walk,696,720.0,31500.0,101002,101001,home,education,True,POINT (841221.23 6517582.12),POINT (841607.6 6518297.5)
93,19,33,93,walk,2609,2700.0,49500.0,101002,101001,home,education,True,POINT (841221.23 6517582.12),POINT (841607.6 6518297.5)
20098,3330,6305,20098,pt,5720,1800.0,30000.0,136003,101001,home,education,True,POINT (847002.0515579024 6519993.825979705),POINT (841607.6 6518297.5)
33050,5443,10716,33050,pt,4240,1800.0,26100.0,214007,101001,home,education,True,POINT (840060.5619822758 6516291.679447936),POINT (841683.2 6517434.1)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,165,309,1167,walk,116,120.0,50400.0,999090,999090,leisure,work,True,None,None
97038,15810,35271,97038,car_passenger,0,300.0,47100.0,999090,999090,leisure,work,True,None,None
80721,12369,27057,80721,car,0,720.0,64800.0,999100,999100,work,home,True,None,None
57935,8530,18193,57935,walk,580,600.0,51600.0,999100,999100,home,work,True,None,None


In [46]:

df_trips = df_trips[(~df_trips["origin_geometry"].isna()) & (~df_trips["destination_geometry"].isna())]

In [52]:
gdf_trips

,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,origin_geometry,destination_geometry,trip_geometry
80797,12383,27096,80797,walk,290,300.0,45000.0,101001,101001,leisure,education,True,POINT (841519.35 6517440.14),POINT (841505.643584 6517298.579645),"LINESTRING (841519.35 6517440.14, 841505.644 6..."
91,19,33,91,walk,696,720.0,31500.0,101002,101001,home,education,True,POINT (841221.23 6517582.12),POINT (841607.6 6518297.5),"LINESTRING (841221.23 6517582.12, 841607.6 651..."
93,19,33,93,walk,2609,2700.0,49500.0,101002,101001,home,education,True,POINT (841221.23 6517582.12),POINT (841607.6 6518297.5),"LINESTRING (841221.23 6517582.12, 841607.6 651..."
20098,3330,6305,20098,pt,5720,1800.0,30000.0,136003,101001,home,education,True,POINT (847002.051558 6519993.82598),POINT (841607.6 6518297.5),"LINESTRING (847002.052 6519993.826, 841607.6 6..."
33050,5443,10716,33050,pt,4240,1800.0,26100.0,214007,101001,home,education,True,POINT (840060.561982 6516291.679448),POINT (841683.2 6517434.1),"LINESTRING (840060.562 6516291.679, 841683.2 6..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98619,16143,36105,98619,car,1090,600.0,63000.0,712451,712551,leisure,shop,True,POINT (881550.51 6542893.89),POINT (880999.00459 6544061.64852),"LINESTRING (881550.51 6542893.89, 880999.005 6..."
99429,16330,36500,99429,car,1090,300.0,36600.0,712451,712551,shop,shop,True,POINT (881004.88 6544012.91),POINT (880999.00459 6544061.64852),"LINESTRING (881004.88 6544012.91, 880999.005 6..."
99472,16340,36521,99472,car_passenger,1090,300.0,36000.0,712451,712551,shop,shop,True,POINT (881225.63 6543765.39),POINT (880999.00459 6544061.64852),"LINESTRING (881225.63 6543765.39, 880999.005 6..."
98785,16182,36199,98785,car,1,300.0,41400.0,712551,712551,work,shop,True,POINT (880999.00459 6544061.64852),POINT (880999.00459 6544061.64852),"LINESTRING (880999.005 6544061.649, 880999.005..."


In [ ]:
df_trips.loc[:, "trip_geometry"] = df_trips.apply(lambda x: geometry.LineString([x["origin_geometry"], x["destination_geometry"]]), axis=1)
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="trip_geometry", crs="EPSG:2154")
gdf_trips["origin_geometry"] = gpd.GeoSeries(gdf_trips["origin_geometry"]).to_wkt()
gdf_trips["destination_geometry"] = gpd.GeoSeries(gdf_trips["destination_geometry"]).to_wkt()

gdf_trips["computed_distance"] = gdf_trips["trip_geometry"].length
gdf_trips["distance_error"] = (gdf_trips["euclidean_distance"] - gdf_trips["computed_distance"]).abs()

In [60]:
gdf_test = gdf_trips.copy()


gdf_test["distance_error"] = gdf_test["euclidean_distance"] - gdf_test["trip_geometry"].length
print("Overall distance error statistics:")
display(gdf_test["distance_error"].describe())

print("Distance error statistics for mode 'car':")
display(gdf_test.loc[gdf_test["mode"] == "car"]["distance_error"].describe())

print("Distance error statistics for mode 'pt':")
display(gdf_test.loc[gdf_test["mode"] == "pt"]["distance_error"].describe())

# Save the result
# count    96047.000000
# mean       -32.873218
# std       1473.856491
# min     -62416.065633
# 25%       -411.774573
# 50%        -20.524282
# 75%        337.584980
# max      51010.000000
# Name: distance_error, dtype: float64

Overall distance error statistics:


count    96047.000000
mean        -8.017674
std       1054.439659
min     -16048.097818
25%       -398.564618
50%         -9.629815
75%        346.297902
max      27474.192845
Name: distance_error, dtype: float64

Distance error statistics for mode 'car':


count    40375.000000
mean       -97.914883
std        851.864805
min      -6714.218168
25%       -460.820893
50%        -47.723583
75%        320.389109
max      18322.487035
Name: distance_error, dtype: float64

Distance error statistics for mode 'pt':


count    12786.000000
mean       531.701113
std       1643.724363
min      -4071.907100
25%       -256.007385
50%        168.601001
75%        842.911814
max      27474.192845
Name: distance_error, dtype: float64

In [68]:


gdf_trips["computed_distance"] = gdf_trips["trip_geometry"].length
gdf_trips["distance_error"] = (gdf_trips["euclidean_distance"] - gdf_trips["computed_distance"]).abs()

In [64]:
import plotly.express as px

fig = px.histogram(
    gdf_test,
    x="distance_error",
    color="mode",
    nbins=100,
    opacity=0.5,
    title="Distance Error Histogram per Mode",
    labels={"distance_error": "Distance Error", "count": "Frequency"},
)
fig.update_layout(barmode="overlay")
fig.show()

In [69]:
gdf_trips.to_parquet("%s/trips.geoparquet" % cleaned_survey_path)

In [71]:
gdf_trips.to_file("%s/trips.geojson" % cleaned_survey_path, driver="GeoJSON")